# 17 — Final annotation and figure consolidation

This notebook reproduces the historical final annotation/figure-consolidation step on the canonical Notebook-15 CNV-consensus branch.

## Historical semantics preserved

The cluster-to-label mappings are unchanged from the historical executed notebook:

- malignant-like clusters: `0, 4, 7, 11`;
- cycling malignant-like candidate: `2`;
- non-malignant/diploid-leaning clusters: `8, 13, 14, 15, 16`;
- review clusters: `1, 3, 5, 6, 9, 10, 12, 17`.

The refined annotation labels, major compartments, analysis-use labels, CopyKAT summaries, marker panel, composition tables, and figure set are preserved.

## Canonical input policy

This notebook requires the standard final H5AD produced by Notebook 15:

`GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad`

Before any output is written, the notebook verifies the canonical historical CopyKAT counts and Notebook-15 final malignancy-group counts.

## Memory-safety policy

The historical notebook loaded the complete full-gene H5AD into memory and wrote two complete H5AD copies. This implementation does not.

- The input H5AD is opened in backed mode.
- Only `.obs`, `X_umap`, variable names, selected color metadata, and the 77 present marker genes are materialized.
- UMAP figures use a lightweight AnnData containing only metadata and `X_umap`.
- The marker dotplot loads only the marker-gene slice.
- The final H5AD is created by sequentially copying the canonical Notebook-15 H5AD and replacing only `/obs` plus the final annotation provenance in `/uns`.
- The compatibility alias is created as a hard link when supported, with a sequential-copy fallback.

No biological decision rule is changed by the memory-safe implementation.

## H5AD validation compatibility

The installed AnnData version can materialize sparse `/layers` even when `read_h5ad(..., backed='r')` is used. Therefore final-output and alias validation are performed directly against `/obs`, `/var`, and `/uns` with h5py/read_elem. This avoids materializing the 739,913,124-entry counts layer. The finalization cell is also recovery-safe: if a completed final H5AD or alias already exists, it is validated and reused rather than copied again.


In [1]:
# =========================
# IMPORTS + PATHS
# =========================
from pathlib import Path
import json
import os
import shutil
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
import h5py

try:
    from anndata.io import write_elem, read_elem
except Exception:
    from anndata._io.specs import write_elem, read_elem


warnings.filterwarnings("ignore")


def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()

    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").exists()
            and (candidate / "metadata").exists()
        ):
            return candidate

    return start


PROJECT_DIR = Path(
    os.environ.get(
        "GLIOMA_PROJECT_ROOT",
        find_repository_root(),
    )
).expanduser().resolve()

PROCESSED_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "integrated"
)

METADATA_DIR = (
    PROJECT_DIR
    / "metadata"
)

FIGURES_DIR = (
    PROJECT_DIR
    / "figures"
    / "final_annotation_CNV_consensus_HVG8000_r0_6"
)

for directory in [
    PROCESSED_DIR,
    METADATA_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


INPUT_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad"
)

FINAL_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation_CNV_consensus.h5ad"
)

FINAL_ALIAS_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_final_annotation.h5ad"
)

CLUSTER_KEY = "leiden_scvi_main"
COPYKAT_KEY = "copykat_pred_clean"
FINAL_GROUP_KEY = "final_malignancy_group_copykat_marker"

print(
    "PROJECT_DIR:",
    PROJECT_DIR,
)
print(
    "INPUT_H5AD:",
    INPUT_H5AD,
)
print(
    "exists:",
    INPUT_H5AD.exists(),
)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 17 requires the canonical Notebook-15 H5AD:\n"
        f"{INPUT_H5AD}"
    )


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad
exists: True


In [2]:
# =========================
# HELPERS
# =========================
def write_tsv_replace(
    df,
    path,
    index=False,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_suffix(
        path.suffix + ".tmp"
    )

    if tmp.exists():
        tmp.unlink()

    df.to_csv(
        tmp,
        sep="\t",
        index=index,
    )

    os.replace(
        tmp,
        path,
    )

    print(
        f"[SAVED] {path} shape={df.shape}"
    )


def choose_first_existing_column(
    df,
    candidates,
    required=True,
    label="column",
):
    for candidate in candidates:
        if candidate in df.columns:
            print(
                f"[FOUND] {label}: {candidate}"
            )
            return candidate

    if required:
        raise KeyError(
            f"None of these {label} candidates found: {candidates}"
        )

    print(
        f"[MISSING] {label}: none of {candidates}"
    )
    return None


def clean_cluster_label(value):
    value = str(value)

    try:
        return str(
            int(
                float(value)
            )
        )
    except Exception:
        return value


def save_current_fig(
    stem,
    dpi=180,
):
    png = (
        FIGURES_DIR
        / f"{stem}.png"
    )

    pdf = (
        FIGURES_DIR
        / f"{stem}.pdf"
    )

    plt.savefig(
        png,
        dpi=dpi,
        bbox_inches="tight",
    )

    plt.savefig(
        pdf,
        bbox_inches="tight",
    )

    plt.close()

    print(
        f"[FIGURE] {png}"
    )

    return (
        str(png),
        str(pdf),
    )


def copy_file_with_progress(
    src,
    dst,
    chunk_size=64 * 1024 * 1024,
):
    src = Path(src)
    dst = Path(dst)

    total = src.stat().st_size
    copied = 0
    next_report = 5

    with src.open("rb") as fin, dst.open("wb") as fout:
        while True:
            block = fin.read(
                chunk_size
            )

            if not block:
                break

            fout.write(
                block
            )

            copied += len(block)

            pct = int(
                copied
                * 100
                / max(
                    total,
                    1,
                )
            )

            if pct >= next_report:
                print(
                    f"  {pct}% "
                    f"({copied / 1024**3:.2f} / "
                    f"{total / 1024**3:.2f} GiB)"
                )

                next_report += 5

        fout.flush()
        os.fsync(
            fout.fileno()
        )

    shutil.copystat(
        src,
        dst,
    )


def replace_h5ad_obs_group(
    h5ad_path,
    obs_df,
):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "obs" in handle:
            del handle["obs"]

        write_elem(
            handle,
            "obs",
            obs_df,
        )

        handle.flush()


def add_h5ad_uns_elements(
    h5ad_path,
    elements,
):
    with h5py.File(
        h5ad_path,
        "r+",
    ) as handle:

        if "uns" not in handle:
            handle.create_group(
                "uns"
            )

        uns_group = handle[
            "uns"
        ]

        for key, value in elements.items():
            if key in uns_group:
                del uns_group[key]

            write_elem(
                uns_group,
                key,
                value,
            )

        handle.flush()


def create_alias(
    source,
    alias,
):
    source = Path(source)
    alias = Path(alias)

    if alias.exists():
        alias.unlink()

    try:
        os.link(
            source,
            alias,
        )

        print(
            "[HARD LINK ALIAS]",
            alias,
        )

        return "hard_link"

    except Exception as error:
        print(
            "[HARD LINK UNAVAILABLE]",
            repr(error),
        )
        print(
            "[COPY ALIAS]"
        )

        copy_file_with_progress(
            source,
            alias,
        )

        return "sequential_copy"


def normalize_copykat_value(value):
    if pd.isna(value):
        return "missing"

    text = str(
        value
    ).strip().lower()

    mapping = {
        "aneuploid": "aneuploid",
        "diploid": "diploid",
        "not.defined": "not_defined",
        "not_defined": "not_defined",
        "missing": "missing",
        "nan": "missing",
    }

    return mapping.get(
        text,
        text,
    )


def counts_dict(series):
    return {
        str(key): int(value)
        for key, value
        in series.value_counts(
            dropna=False
        ).items()
    }

def validate_h5ad_metadata_only(
    h5ad_path,
    expected_shape,
    expected_major_counts,
    require_provenance=True,
):
    """
    Validate an H5AD without asking AnnData to read X or layers.

    This is intentionally h5py/read_elem based because the installed
    AnnData version may materialize sparse layers even with backed='r'.
    """
    h5ad_path = Path(h5ad_path)

    with h5py.File(
        h5ad_path,
        "r",
    ) as handle:

        if "obs" not in handle:
            raise RuntimeError(
                f"{h5ad_path} lacks /obs."
            )

        if "var" not in handle:
            raise RuntimeError(
                f"{h5ad_path} lacks /var."
            )

        obs_check = read_elem(
            handle["obs"]
        )

        var_check = read_elem(
            handle["var"]
        )

        observed_shape = (
            int(
                obs_check.shape[
                    0
                ]
            ),
            int(
                var_check.shape[
                    0
                ]
            ),
        )

        if observed_shape != expected_shape:
            raise RuntimeError(
                f"H5AD metadata-only shape mismatch for {h5ad_path}: "
                f"{observed_shape} != {expected_shape}"
            )

        required_final_cols = [
            "final_annotation_refined",
            "final_compartment_major",
            "final_analysis_use",
            "final_compartment_simple",
        ]

        missing_final_cols = [
            column
            for column in required_final_cols
            if column not in obs_check.columns
        ]

        if missing_final_cols:
            raise RuntimeError(
                f"{h5ad_path} is missing required final-annotation columns: "
                + ", ".join(
                    missing_final_cols
                )
            )

        final_major_counts = counts_dict(
            obs_check[
                "final_compartment_major"
            ]
        )

        if final_major_counts != expected_major_counts:
            raise RuntimeError(
                f"{h5ad_path} major-compartment checkpoint failed. "
                f"Observed: {final_major_counts}"
            )

        if require_provenance:
            if "uns" not in handle:
                raise RuntimeError(
                    f"{h5ad_path} lacks /uns."
                )

            if (
                "final_annotation_consolidation"
                not in handle[
                    "uns"
                ]
            ):
                raise RuntimeError(
                    f"{h5ad_path} lacks "
                    "/uns/final_annotation_consolidation provenance."
                )

    return {
        "path": str(
            h5ad_path
        ),
        "shape": observed_shape,
        "major_counts": final_major_counts,
    }


In [3]:
# =========================
# HISTORICAL MARKER PANEL
# =========================
marker_panel = {
    "malignant_stem_like": [
        "SOX2",
        "NES",
        "PROM1",
        "CD44",
        "VIM",
    ],
    "OPC_NPC_like": [
        "PTPRZ1",
        "BCAN",
        "OLIG1",
        "OLIG2",
        "PDGFRA",
        "SOX4",
        "DLL3",
    ],
    "AC_like_reactive": [
        "GFAP",
        "AQP4",
        "ALDOC",
        "CLU",
        "SPARCL1",
        "SLC1A3",
    ],
    "hypoxia_stress": [
        "CA9",
        "VEGFA",
        "HILPDA",
        "LOX",
        "ADM",
        "NDRG1",
    ],
    "cycling": [
        "MKI67",
        "TOP2A",
        "UBE2C",
        "BIRC5",
        "PBK",
        "CENPF",
    ],
    "myeloid_TAM": [
        "C1QA",
        "C1QB",
        "C1QC",
        "TREM2",
        "APOE",
        "CSF1R",
        "AIF1",
        "TYROBP",
        "CD68",
    ],
    "monocyte_inflammatory": [
        "FCN1",
        "S100A8",
        "S100A9",
        "LYZ",
        "LST1",
        "AQP9",
    ],
    "T_NK": [
        "CD3D",
        "CD3E",
        "TRAC",
        "NKG7",
        "GZMA",
        "GNLY",
        "LCK",
    ],
    "oligodendrocyte": [
        "MOG",
        "MAG",
        "MOBP",
        "PLP1",
        "ERMN",
    ],
    "endothelial": [
        "PECAM1",
        "VWF",
        "KDR",
        "ERG",
        "ESAM",
        "EMCN",
        "CLEC14A",
    ],
    "mural_pericyte": [
        "PDGFRB",
        "RGS5",
        "ACTA2",
        "MCAM",
        "ENPEP",
        "CD248",
    ],
    "B_plasma": [
        "CD79A",
        "CD79B",
        "MS4A1",
        "MZB1",
        "JCHAIN",
        "IGHM",
    ],
    "erythroid_Hb": [
        "HBA1",
        "HBA2",
        "HBB",
        "ALAS2",
    ],
}


In [4]:
# =========================
# BACKED INPUT READ — OBS, UMAP, VAR NAMES, MARKER SLICE ONLY
# =========================
backed = sc.read_h5ad(
    INPUT_H5AD,
    backed="r",
)

try:
    source_shape = backed.shape

    obs = (
        backed.obs
        .copy()
    )

    obs.index = (
        obs.index
        .astype(str)
    )

    var_names = (
        backed.var_names
        .astype(str)
        .tolist()
    )

    obsm_keys = list(
        backed.obsm.keys()
    )

    layer_keys = list(
        backed.layers.keys()
    )

    if "X_umap" not in obsm_keys:
        raise KeyError(
            "X_umap not found in canonical Notebook-15 H5AD."
        )

    source_umap = np.asarray(
        backed.obsm[
            "X_umap"
        ]
    ).copy()

    required_cols = [
        CLUSTER_KEY,
        COPYKAT_KEY,
        FINAL_GROUP_KEY,
    ]

    missing = [
        column
        for column in required_cols
        if column not in obs.columns
    ]

    if missing:
        raise KeyError(
            f"Missing required columns in obs: {missing}"
        )

    CONDITION_KEY = choose_first_existing_column(
        obs,
        [
            "condition",
            "disease_condition",
            "diagnosis",
            "group",
        ],
        False,
        "condition",
    )

    DATASET_KEY = choose_first_existing_column(
        obs,
        [
            "core_dataset",
            "dataset_id",
            "dataset",
            "study",
        ],
        False,
        "dataset",
    )

    SAMPLE_KEY = choose_first_existing_column(
        obs,
        [
            "sample_id",
            "sample",
            "orig.ident",
            "sample_name",
        ],
        False,
        "sample",
    )

    PATIENT_KEY = choose_first_existing_column(
        obs,
        [
            "patient_id",
            "patient",
            "donor_id",
            "case_id",
        ],
        False,
        "patient",
    )

    # Historical candidate list intentionally preserved.
    # It did not include annotation_marker_based_HVG8000_r0_6.
    MARKER_ANNOT_KEY = choose_first_existing_column(
        obs,
        [
            "annotation_marker_based",
            "marker_based_annotation",
            "cluster_marker_annotation",
            "annotation",
            "cell_type_marker_based",
        ],
        False,
        "marker annotation",
    )

    marker_genes_present = []
    varname_set = set(
        var_names
    )

    for panel in marker_panel:
        for gene in marker_panel[
            panel
        ]:
            if (
                gene in varname_set
                and gene not in marker_genes_present
            ):
                marker_genes_present.append(
                    gene
                )

    if len(
        marker_genes_present
    ) > 0:
        print(
            "[LOAD MARKER SLICE]",
            len(
                marker_genes_present
            ),
            "genes",
        )

        marker_adata = (
            backed[
                :,
                marker_genes_present,
            ]
            .to_memory()
        )

    else:
        marker_adata = None

    color_uns = {}

    for key in [
        "condition_colors",
        "core_dataset_colors",
        "copykat_pred_clean_colors",
        "leiden_scvi_main_colors",
    ]:
        if key in backed.uns:
            color_uns[
                key
            ] = np.asarray(
                backed.uns[
                    key
                ]
            ).copy()

finally:
    backed.file.close()


print(
    "Source shape:",
    source_shape,
)
print(
    "obs columns:",
    len(
        obs.columns
    ),
)
print(
    "var columns: 0"
)
print(
    "obsm keys:",
    obsm_keys,
)
print(
    "layer keys:",
    layer_keys,
)
print(
    "UMAP shape:",
    source_umap.shape,
)
print(
    "marker genes present:",
    len(
        marker_genes_present
    ),
)


if source_shape != (
    351427,
    15176,
):
    raise RuntimeError(
        f"Canonical Notebook-15 H5AD shape mismatch: {source_shape}"
    )

if source_umap.shape != (
    351427,
    2,
):
    raise RuntimeError(
        f"Canonical UMAP shape mismatch: {source_umap.shape}"
    )


obs[
    CLUSTER_KEY
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        clean_cluster_label
    )
    .astype(
        "category"
    )
)

obs[
    COPYKAT_KEY
] = (
    obs[
        COPYKAT_KEY
    ]
    .astype(str)
    .astype(
        "category"
    )
)

obs[
    FINAL_GROUP_KEY
] = (
    obs[
        FINAL_GROUP_KEY
    ]
    .astype(str)
    .astype(
        "category"
    )
)


EXPECTED_COPYKAT_COUNTS = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not_defined": 37037,
    "missing": 2,
}

observed_copykat_counts = counts_dict(
    obs[
        COPYKAT_KEY
    ]
    .map(
        normalize_copykat_value
    )
)

if observed_copykat_counts != EXPECTED_COPYKAT_COUNTS:
    raise RuntimeError(
        "Notebook 17 is not consuming the exact canonical historical "
        f"CopyKAT calls. Observed: {observed_copykat_counts}"
    )


EXPECTED_FINAL_GROUP_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

observed_final_group_counts = counts_dict(
    obs[
        FINAL_GROUP_KEY
    ]
)

if observed_final_group_counts != EXPECTED_FINAL_GROUP_COUNTS:
    raise RuntimeError(
        "Notebook-15 final malignancy-group checkpoint failed. "
        f"Observed: {observed_final_group_counts}"
    )


if len(
    marker_genes_present
) != 77:
    raise RuntimeError(
        "Historical marker-gene presence checkpoint failed: "
        f"{len(marker_genes_present)} != 77"
    )


print(
    "PASS: canonical Notebook-15 and marker-presence input checkpoints match."
)


[FOUND] condition: condition
[FOUND] dataset: core_dataset
[FOUND] sample: sample_id
[FOUND] patient: patient_id
[MISSING] marker annotation: none of ['annotation_marker_based', 'marker_based_annotation', 'cluster_marker_annotation', 'annotation', 'cell_type_marker_based']
[LOAD MARKER SLICE] 77 genes
Source shape: (351427, 15176)
obs columns: 88
var columns: 0
obsm keys: ['X_scVI', 'X_umap']
layer keys: ['counts']
UMAP shape: (351427, 2)
marker genes present: 77
PASS: canonical Notebook-15 and marker-presence input checkpoints match.


In [5]:
# =========================
# HISTORICAL FINAL ANNOTATION MAPPINGS
# =========================
cluster_to_final_annotation = {
    "0":  "malignant_OPC_NPC_like_CNV_supported",
    "1":  "review_myeloid_C1Q_TREM2_APOE_TAM_mixed_CNV",
    "2":  "cycling_malignant_like_candidate_CNV_supported",
    "3":  "review_myeloid_inflammatory_MES_associated_mixed_CNV",
    "4":  "malignant_hypoxia_stress_CNV_supported",
    "5":  "review_T_NK_cells_dataset_biased_mixed_CNV",
    "6":  "review_myeloid_APC_HLA_CD163_TAM_mixed_CNV",
    "7":  "malignant_AC_like_reactive_CNV_supported",
    "8":  "non_malignant_oligodendrocyte_myelinating_reference_like",
    "9":  "review_ambiguous_mixed_Hb_T_NK_HLA",
    "10": "review_myeloid_FCN1_inflammatory_monocyte_CNV_discordant",
    "11": "malignant_NPC_neuronal_like_CNV_supported",
    "12": "review_MES_ECM_stromal_like_ambiguous_CNV_high",
    "13": "non_malignant_mural_pericyte_fibroblast_like",
    "14": "non_malignant_endothelial_reference_like",
    "15": "non_malignant_B_plasma_like",
    "16": "non_malignant_rare_mast_cell_like_low_count",
    "17": "review_erythroid_Hb_high_or_ambient_RBC",
}


cluster_to_major_compartment = {
    "0": "malignant_like",
    "4": "malignant_like",
    "7": "malignant_like",
    "11": "malignant_like",

    "2": "cycling_malignant_like_candidate",

    "8": "non_malignant_or_diploid_leaning",
    "13": "non_malignant_or_diploid_leaning",
    "14": "non_malignant_or_diploid_leaning",
    "15": "non_malignant_or_diploid_leaning",
    "16": "non_malignant_or_diploid_leaning",

    "1": "review",
    "3": "review",
    "5": "review",
    "6": "review",
    "9": "review",
    "10": "review",
    "12": "review",
    "17": "review",
}


cluster_to_analysis_use = {
    "0": "use_malignant",
    "4": "use_malignant",
    "7": "use_malignant",
    "11": "use_malignant",

    "2": "use_cycling_candidate_separately",

    "8": "use_reference_non_malignant",
    "14": "use_reference_non_malignant",

    "13": "use_non_malignant_with_caution",
    "15": "use_non_malignant_with_caution",
    "16": "use_non_malignant_with_caution_low_count",

    "1": "review_exclude_from_core_malignant",
    "3": "review_exclude_from_core_malignant",
    "5": "review_exclude_from_core_malignant",
    "6": "review_exclude_from_core_malignant",
    "9": "review_exclude_from_core_malignant",
    "10": "review_exclude_from_core_malignant_CNV_discordant",
    "12": "review_exclude_from_core_malignant_ambiguous_CNV_high",
    "17": "review_exclude_low_count_Hb_high",
}


obs[
    "final_annotation_refined"
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        cluster_to_final_annotation
    )
    .fillna(
        "unmapped_cluster_review"
    )
    .astype(
        "category"
    )
)


obs[
    "final_compartment_major"
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        cluster_to_major_compartment
    )
    .fillna(
        "review"
    )
    .astype(
        "category"
    )
)


obs[
    "final_analysis_use"
] = (
    obs[
        CLUSTER_KEY
    ]
    .astype(str)
    .map(
        cluster_to_analysis_use
    )
    .fillna(
        "review"
    )
    .astype(
        "category"
    )
)


simple_map = {
    "malignant_like": "malignant_like",
    "cycling_malignant_like_candidate": "cycling_candidate",
    "non_malignant_or_diploid_leaning": "non_malignant_diploid_leaning",
    "review": "review_ambiguous_discordant",
}


obs[
    "final_compartment_simple"
] = (
    obs[
        "final_compartment_major"
    ]
    .astype(str)
    .map(
        simple_map
    )
    .astype(
        "category"
    )
)


EXPECTED_MAJOR_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

observed_major_counts = counts_dict(
    obs[
        "final_compartment_major"
    ]
)

if observed_major_counts != EXPECTED_MAJOR_COUNTS:
    raise RuntimeError(
        "Historical final-compartment mapping checkpoint failed. "
        f"Observed: {observed_major_counts}"
    )


if obs[
    "final_annotation_refined"
].nunique() != 18:
    raise RuntimeError(
        "Historical refined-annotation checkpoint failed: expected 18 labels."
    )


display(
    obs[
        [
            "final_annotation_refined",
            "final_compartment_major",
            "final_analysis_use",
            "final_compartment_simple",
        ]
    ].head()
)

print(
    "PASS: historical cluster-to-final-annotation mappings reproduced exactly."
)


,final_annotation_refined,final_compartment_major,final_analysis_use,final_compartment_simple
DS1_GSE103224:PJ016_cell000216,malignant_AC_like_reactive_CNV_supported,malignant_like,use_malignant,malignant_like
DS1_GSE103224:PJ016_cell000217,malignant_AC_like_reactive_CNV_supported,malignant_like,use_malignant,malignant_like
DS1_GSE103224:PJ016_cell000218,malignant_AC_like_reactive_CNV_supported,malignant_like,use_malignant,malignant_like
DS1_GSE103224:PJ016_cell000219,cycling_malignant_like_candidate_CNV_supported,cycling_malignant_like_candidate,use_cycling_candidate_separately,cycling_candidate
DS1_GSE103224:PJ016_cell000220,malignant_AC_like_reactive_CNV_supported,malignant_like,use_malignant,malignant_like


PASS: historical cluster-to-final-annotation mappings reproduced exactly.


In [6]:
# =========================
# READINESS + CONSOLIDATED CELL COUNTS
# =========================
readiness = pd.DataFrame(
    [
        {
            "metric": "input_h5ad",
            "value": str(
                INPUT_H5AD
            ),
        },
        {
            "metric": "n_cells",
            "value": int(
                source_shape[
                    0
                ]
            ),
        },
        {
            "metric": "n_genes",
            "value": int(
                source_shape[
                    1
                ]
            ),
        },
        {
            "metric": "cluster_key",
            "value": CLUSTER_KEY,
        },
        {
            "metric": "copykat_key",
            "value": COPYKAT_KEY,
        },
        {
            "metric": "final_group_key_previous",
            "value": FINAL_GROUP_KEY,
        },
        {
            "metric": "has_X_umap",
            "value": (
                "X_umap"
                in obsm_keys
            ),
        },
        {
            "metric": "has_X_scVI",
            "value": (
                "X_scVI"
                in obsm_keys
            ),
        },
        {
            "metric": "final_annotation_refined_created",
            "value": (
                "final_annotation_refined"
                in obs.columns
            ),
        },
    ]
)


READINESS_TSV = (
    METADATA_DIR
    / "final_annotation_consolidation_readiness_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    readiness,
    READINESS_TSV,
)


cell_counts = []

for key in [
    CLUSTER_KEY,
    "final_annotation_refined",
    "final_compartment_major",
    "final_compartment_simple",
    "final_analysis_use",
    COPYKAT_KEY,
]:
    vc = (
        obs[
            key
        ]
        .astype(str)
        .value_counts(
            dropna=False
        )
        .rename_axis(
            "label"
        )
        .reset_index(
            name="n_cells"
        )
    )

    vc[
        "fraction"
    ] = (
        vc[
            "n_cells"
        ]
        / source_shape[
            0
        ]
    )

    vc[
        "category"
    ] = key

    cell_counts.append(
        vc
    )


cell_counts_df = pd.concat(
    cell_counts,
    ignore_index=True,
)


CELL_COUNTS_TSV = (
    METADATA_DIR
    / "final_annotation_consolidated_cell_counts_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    cell_counts_df,
    CELL_COUNTS_TSV,
)


if readiness.shape != (
    9,
    2,
):
    raise RuntimeError(
        f"Historical readiness checkpoint failed: {readiness.shape} != (9, 2)"
    )

if cell_counts_df.shape != (
    57,
    4,
):
    raise RuntimeError(
        "Historical consolidated cell-count checkpoint failed: "
        f"{cell_counts_df.shape} != (57, 4)"
    )


display(
    cell_counts_df.head(
        20
    )
)

print(
    "PASS: historical readiness and cell-count table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidation_readiness_HVG8000_r0_6.tsv shape=(9, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidated_cell_counts_HVG8000_r0_6.tsv shape=(57, 4)


,label,n_cells,fraction,category
0,0,61646,0.175416,leiden_scvi_main
1,1,59791,0.170138,leiden_scvi_main
2,2,38634,0.109935,leiden_scvi_main
3,3,35355,0.100604,leiden_scvi_main
4,4,32896,0.093607,leiden_scvi_main
5,5,23977,0.068228,leiden_scvi_main
6,6,22943,0.065285,leiden_scvi_main
7,7,22525,0.064096,leiden_scvi_main
8,8,14809,0.042140,leiden_scvi_main
9,9,13864,0.039451,leiden_scvi_main


PASS: historical readiness and cell-count table shapes match exactly.


In [7]:
# =========================
# HISTORICAL CLUSTER SUMMARY
# =========================
summary_obs = (
    obs
    .copy()
)

summary_obs[
    "_cluster"
] = (
    summary_obs[
        CLUSTER_KEY
    ]
    .astype(str)
)


rows = []

for cluster, sub in summary_obs.groupby(
    "_cluster",
    observed=True,
):
    row = {
        "cluster": cluster,
        "n_cells": int(
            sub.shape[
                0
            ]
        ),
        "fraction_total": float(
            sub.shape[
                0
            ]
            / summary_obs.shape[
                0
            ]
        ),
        "final_annotation_refined": (
            sub[
                "final_annotation_refined"
            ]
            .astype(str)
            .mode()
            .iloc[
                0
            ]
        ),
        "final_compartment_major": (
            sub[
                "final_compartment_major"
            ]
            .astype(str)
            .mode()
            .iloc[
                0
            ]
        ),
        "final_analysis_use": (
            sub[
                "final_analysis_use"
            ]
            .astype(str)
            .mode()
            .iloc[
                0
            ]
        ),
    }

    if MARKER_ANNOT_KEY is not None:
        row[
            "original_marker_annotation_mode"
        ] = (
            sub[
                MARKER_ANNOT_KEY
            ]
            .astype(str)
            .mode()
            .iloc[
                0
            ]
        )

    copy_counts = (
        sub[
            COPYKAT_KEY
        ]
        .astype(str)
        .value_counts()
    )

    defined = int(
        copy_counts.get(
            "aneuploid",
            0,
        )
        + copy_counts.get(
            "diploid",
            0,
        )
    )

    row[
        "copykat_aneuploid_n"
    ] = int(
        copy_counts.get(
            "aneuploid",
            0,
        )
    )

    row[
        "copykat_diploid_n"
    ] = int(
        copy_counts.get(
            "diploid",
            0,
        )
    )

    row[
        "copykat_not_defined_n"
    ] = int(
        copy_counts.get(
            "not.defined",
            0,
        )
        + copy_counts.get(
            "not_defined",
            0,
        )
    )

    row[
        "copykat_defined_n"
    ] = defined

    row[
        "copykat_aneuploid_fraction_defined"
    ] = (
        float(
            row[
                "copykat_aneuploid_n"
            ]
            / defined
        )
        if defined > 0
        else np.nan
    )

    if CONDITION_KEY:
        row[
            "top_condition"
        ] = (
            sub[
                CONDITION_KEY
            ]
            .astype(str)
            .value_counts()
            .idxmax()
        )

        row[
            "top_condition_fraction"
        ] = float(
            sub[
                CONDITION_KEY
            ]
            .astype(str)
            .value_counts(
                normalize=True
            )
            .max()
        )

    if DATASET_KEY:
        row[
            "top_dataset"
        ] = (
            sub[
                DATASET_KEY
            ]
            .astype(str)
            .value_counts()
            .idxmax()
        )

        row[
            "top_dataset_fraction"
        ] = float(
            sub[
                DATASET_KEY
            ]
            .astype(str)
            .value_counts(
                normalize=True
            )
            .max()
        )

    if SAMPLE_KEY:
        row[
            "top_sample"
        ] = (
            sub[
                SAMPLE_KEY
            ]
            .astype(str)
            .value_counts()
            .idxmax()
        )

        row[
            "top_sample_fraction"
        ] = float(
            sub[
                SAMPLE_KEY
            ]
            .astype(str)
            .value_counts(
                normalize=True
            )
            .max()
        )

    if PATIENT_KEY:
        row[
            "top_patient"
        ] = (
            sub[
                PATIENT_KEY
            ]
            .astype(str)
            .value_counts()
            .idxmax()
        )

        row[
            "top_patient_fraction"
        ] = float(
            sub[
                PATIENT_KEY
            ]
            .astype(str)
            .value_counts(
                normalize=True
            )
            .max()
        )

    rows.append(
        row
    )


cluster_summary = pd.DataFrame(
    rows
)

cluster_summary[
    "cluster_numeric"
] = pd.to_numeric(
    cluster_summary[
        "cluster"
    ],
    errors="coerce",
)

cluster_summary = (
    cluster_summary
    .sort_values(
        [
            "cluster_numeric",
            "cluster",
        ]
    )
    .drop(
        columns=[
            "cluster_numeric"
        ]
    )
)


CLUSTER_SUMMARY_TSV = (
    METADATA_DIR
    / "final_annotation_consolidated_cluster_summary_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    cluster_summary,
    CLUSTER_SUMMARY_TSV,
)


if cluster_summary.shape != (
    18,
    19,
):
    raise RuntimeError(
        "Historical consolidated cluster-summary checkpoint failed: "
        f"{cluster_summary.shape} != (18, 19)"
    )


display(
    cluster_summary
)

print(
    "PASS: historical consolidated cluster-summary shape matches exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidated_cluster_summary_HVG8000_r0_6.tsv shape=(18, 19)


,cluster,n_cells,fraction_total,final_annotation_refined,final_compartment_major,final_analysis_use,copykat_aneuploid_n,copykat_diploid_n,copykat_not_defined_n,copykat_defined_n,copykat_aneuploid_fraction_defined,top_condition,top_condition_fraction,top_dataset,top_dataset_fraction,top_sample,top_sample_fraction,top_patient,top_patient_fraction
0,0,61646,0.175416,malignant_OPC_NPC_like_CNV_supported,malignant_like,use_malignant,45910,8947,6789,54857,0.836903,GBM,0.444100,DS3_GSE182109,0.555900,GSM5518602_ndGBM-01-D,0.064692,04,0.185527
1,1,59791,0.170138,review_myeloid_C1Q_TREM2_APOE_TAM_mixed_CNV,review,review_exclude_from_core_malignant,29402,24843,5546,54245,0.542022,ndGBM,0.426586,DS3_GSE182109,0.669649,GSM5518603_ndGBM-01-F,0.062953,01,0.148534
10,2,38634,0.109935,cycling_malignant_like_candidate_CNV_supported,cycling_malignant_like_candidate,use_cycling_candidate_separately,29096,5913,3625,35009,0.831101,GBM,0.447326,DS3_GSE182109,0.552674,GSM5518621_rGBM-04-3,0.057203,04,0.240358
11,3,35355,0.100604,review_myeloid_inflammatory_MES_associated_mix...,review,review_exclude_from_core_malignant,21350,12033,1971,33383,0.639547,rGBM,0.586225,DS3_GSE182109,0.746599,GSM5518597_rGBM-01-B,0.217282,01,0.470117
12,4,32896,0.093607,malignant_hypoxia_stress_CNV_supported,malignant_like,use_malignant,24754,4562,3580,29316,0.844385,GBM,0.606548,DS4_GSE278450,0.466652,GSM8546833_BT356,0.149471,01,0.151751
13,5,23977,0.068228,review_T_NK_cells_dataset_biased_mixed_CNV,review,review_exclude_from_core_malignant,8354,12387,3236,20741,0.402777,rGBM,0.617091,DS3_GSE182109,0.967052,GSM5518615_rGBM-02-5,0.194603,02,0.555824
14,6,22943,0.065285,review_myeloid_APC_HLA_CD163_TAM_mixed_CNV,review,review_exclude_from_core_malignant,10847,10728,1368,21575,0.502758,GBM,0.366517,DS3_GSE182109,0.633483,GSM5518636_ndGBM-07,0.089134,02,0.157259
15,7,22525,0.064096,malignant_AC_like_reactive_CNV_supported,malignant_like,use_malignant,14292,5140,3092,19432,0.735488,GBM,0.614162,DS3_GSE182109,0.385838,GSM8546834_BT363,0.120844,04,0.121154
16,8,14809,0.042140,non_malignant_oligodendrocyte_myelinating_refe...,non_malignant_or_diploid_leaning,use_reference_non_malignant,2787,11317,705,14104,0.197604,GBM,0.610777,DS4_GSE278450,0.558714,GSM8546845_BT396,0.113445,04,0.179485
17,9,13864,0.039451,review_ambiguous_mixed_Hb_T_NK_HLA,review,review_exclude_from_core_malignant,3400,6255,4209,9655,0.352149,ndGBM,0.806621,DS3_GSE182109,0.941864,GSM5518629_ndGBM-10,0.696336,10,0.696336


PASS: historical consolidated cluster-summary shape matches exactly.


In [8]:
# =========================
# CONDITION / DATASET / SAMPLE COMPOSITION
# =========================
composition_paths = {}


if CONDITION_KEY:
    cond_counts = (
        obs.groupby(
            [
                CONDITION_KEY,
                "final_compartment_major",
            ],
            observed=True,
        )
        .size()
        .reset_index(
            name="n_cells"
        )
    )

    cond_counts[
        "condition_total_cells"
    ] = (
        cond_counts.groupby(
            CONDITION_KEY
        )[
            "n_cells"
        ]
        .transform(
            "sum"
        )
    )

    cond_counts[
        "fraction_within_condition"
    ] = (
        cond_counts[
            "n_cells"
        ]
        / cond_counts[
            "condition_total_cells"
        ]
    )

    path = (
        METADATA_DIR
        / "final_annotation_condition_compartment_counts_HVG8000_r0_6.tsv"
    )

    write_tsv_replace(
        cond_counts,
        path,
    )

    composition_paths[
        "condition_compartment_counts"
    ] = str(
        path
    )


if DATASET_KEY:
    dataset_counts = (
        obs.groupby(
            [
                DATASET_KEY,
                "final_compartment_major",
            ],
            observed=True,
        )
        .size()
        .reset_index(
            name="n_cells"
        )
    )

    dataset_counts[
        "dataset_total_cells"
    ] = (
        dataset_counts.groupby(
            DATASET_KEY
        )[
            "n_cells"
        ]
        .transform(
            "sum"
        )
    )

    dataset_counts[
        "fraction_within_dataset"
    ] = (
        dataset_counts[
            "n_cells"
        ]
        / dataset_counts[
            "dataset_total_cells"
        ]
    )

    path = (
        METADATA_DIR
        / "final_annotation_dataset_compartment_counts_HVG8000_r0_6.tsv"
    )

    write_tsv_replace(
        dataset_counts,
        path,
    )

    composition_paths[
        "dataset_compartment_counts"
    ] = str(
        path
    )


if SAMPLE_KEY:
    group_cols = [
        SAMPLE_KEY
    ]

    for key in [
        PATIENT_KEY,
        CONDITION_KEY,
        DATASET_KEY,
    ]:
        if (
            key
            and key not in group_cols
        ):
            group_cols.append(
                key
            )

    group_cols.append(
        "final_compartment_major"
    )

    sample_counts = (
        obs.groupby(
            group_cols,
            observed=True,
        )
        .size()
        .reset_index(
            name="n_cells"
        )
    )

    sample_counts[
        "sample_total_cells"
    ] = (
        sample_counts.groupby(
            SAMPLE_KEY
        )[
            "n_cells"
        ]
        .transform(
            "sum"
        )
    )

    sample_counts[
        "fraction_within_sample"
    ] = (
        sample_counts[
            "n_cells"
        ]
        / sample_counts[
            "sample_total_cells"
        ]
    )

    path = (
        METADATA_DIR
        / "final_annotation_sample_level_compartment_composition_HVG8000_r0_6.tsv"
    )

    write_tsv_replace(
        sample_counts,
        path,
    )

    composition_paths[
        "sample_level_compartment_composition"
    ] = str(
        path
    )


EXPECTED_COMPOSITION_SHAPES = {
    "condition": (16, 5),
    "dataset": (16, 5),
    "sample": (319, 8),
}

observed_composition_shapes = {
    "condition": (
        cond_counts.shape
        if CONDITION_KEY
        else None
    ),
    "dataset": (
        dataset_counts.shape
        if DATASET_KEY
        else None
    ),
    "sample": (
        sample_counts.shape
        if SAMPLE_KEY
        else None
    ),
}

if (
    observed_composition_shapes
    != EXPECTED_COMPOSITION_SHAPES
):
    raise RuntimeError(
        "Historical final-annotation composition-table checkpoint failed. "
        f"Observed: {observed_composition_shapes}"
    )


print(
    json.dumps(
        composition_paths,
        indent=2,
    )
)

print(
    "PASS: historical composition-table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_condition_compartment_counts_HVG8000_r0_6.tsv shape=(16, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_dataset_compartment_counts_HVG8000_r0_6.tsv shape=(16, 5)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_sample_level_compartment_composition_HVG8000_r0_6.tsv shape=(319, 8)
{
  "condition_compartment_counts": "G:\\Repository\\glioma-cnv-single-cell-atlas\\metadata\\final_annotation_condition_compartment_counts_HVG8000_r0_6.tsv",
  "dataset_compartment_counts": "G:\\Repository\\glioma-cnv-single-cell-atlas\\metadata\\final_annotation_dataset_compartment_counts_HVG8000_r0_6.tsv",
  "sample_level_compartment_composition": "G:\\Repository\\glioma-cnv-single-cell-atlas\\metadata\\final_annotation_sample_level_compartment_composition_HVG8000_r0_6.tsv"
}
PASS: historical composition-table shapes match exactly.


In [9]:
# =========================
# MARKER PANEL PRESENCE
# =========================
rows = []

varname_set = set(
    var_names
)

for panel, genes in marker_panel.items():
    for gene in genes:
        rows.append(
            {
                "panel": panel,
                "gene": gene,
                "present": (
                    gene
                    in varname_set
                ),
            }
        )


marker_presence = pd.DataFrame(
    rows
)

MARKER_PRESENCE_TSV = (
    METADATA_DIR
    / "final_annotation_marker_panel_presence_HVG8000_r0_6.tsv"
)

write_tsv_replace(
    marker_presence,
    MARKER_PRESENCE_TSV,
)


if marker_presence.shape != (
    80,
    3,
):
    raise RuntimeError(
        "Historical marker-panel presence table checkpoint failed: "
        f"{marker_presence.shape} != (80, 3)"
    )


if int(
    marker_presence[
        "present"
    ].sum()
) != 77:
    raise RuntimeError(
        "Historical marker-panel present-gene checkpoint failed: "
        f"{int(marker_presence['present'].sum())} != 77"
    )


display(
    marker_presence.groupby(
        "panel"
    )[
        "present"
    ]
    .agg(
        [
            "sum",
            "count",
        ]
    )
)

print(
    "PASS: historical marker-panel presence checkpoint matches exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_marker_panel_presence_HVG8000_r0_6.tsv shape=(80, 3)


,sum,count
panel,,
AC_like_reactive,6,6
B_plasma,4,6
OPC_NPC_like,7,7
T_NK,7,7
cycling,6,6
endothelial,7,7
erythroid_Hb,3,4
hypoxia_stress,6,6
malignant_stem_like,5,5


PASS: historical marker-panel presence checkpoint matches exactly.


In [10]:
# =========================
# UMAP FIGURES FROM LIGHTWEIGHT OBJECT
# =========================
sc.settings.figdir = str(
    FIGURES_DIR
)

sc.set_figure_params(
    figsize=(
        7,
        6,
    ),
    dpi=120,
    frameon=False,
)


plot_obs = (
    obs
    .copy()
)

plot_adata = ad.AnnData(
    X=np.empty(
        (
            len(
                plot_obs
            ),
            0,
        ),
        dtype=np.float32,
    ),
    obs=plot_obs,
)

plot_adata.obsm[
    "X_umap"
] = source_umap


for key, value in color_uns.items():
    plot_adata.uns[
        key
    ] = value


figure_records = []


umap_keys = [
    (
        "final_compartment_major",
        "UMAP_final_compartment_major",
    ),
    (
        "final_annotation_refined",
        "UMAP_final_annotation_refined",
    ),
    (
        "final_analysis_use",
        "UMAP_final_analysis_use",
    ),
    (
        COPYKAT_KEY,
        "UMAP_CopyKAT_call",
    ),
    (
        CLUSTER_KEY,
        "UMAP_leiden_scvi_main",
    ),
]


if CONDITION_KEY:
    umap_keys.append(
        (
            CONDITION_KEY,
            "UMAP_condition",
        )
    )


if DATASET_KEY:
    umap_keys.append(
        (
            DATASET_KEY,
            "UMAP_dataset",
        )
    )


for key, stem in umap_keys:
    print(
        "[PLOT]",
        key,
    )

    sc.pl.umap(
        plot_adata,
        color=key,
        legend_loc="right margin",
        size=2,
        show=False,
        title=key,
    )

    png, pdf = save_current_fig(
        f"final_annotation_{stem}_HVG8000_r0_6"
    )

    figure_records.append(
        {
            "figure": stem,
            "type": "umap",
            "color_key": key,
            "png": png,
            "pdf": pdf,
        }
    )


FIGURE_MANIFEST_TSV = (
    METADATA_DIR
    / "final_annotation_consolidation_figure_manifest_HVG8000_r0_6.tsv"
)

figure_manifest = pd.DataFrame(
    figure_records
)

write_tsv_replace(
    figure_manifest,
    FIGURE_MANIFEST_TSV,
)


if figure_manifest.shape != (
    7,
    5,
):
    raise RuntimeError(
        "Historical initial figure-manifest checkpoint failed: "
        f"{figure_manifest.shape} != (7, 5)"
    )


generated_color_uns = {}

for key in [
    "final_compartment_major_colors",
    "final_annotation_refined_colors",
    "final_analysis_use_colors",
]:
    if key in plot_adata.uns:
        generated_color_uns[
            key
        ] = np.asarray(
            plot_adata.uns[
                key
            ]
        ).copy()


print(
    "PASS: historical seven-UMAP figure manifest reproduced."
)


[PLOT] final_compartment_major
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_UMAP_final_compartment_major_HVG8000_r0_6.png
[PLOT] final_annotation_refined
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_UMAP_final_annotation_refined_HVG8000_r0_6.png
[PLOT] final_analysis_use
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_UMAP_final_analysis_use_HVG8000_r0_6.png
[PLOT] copykat_pred_clean
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_UMAP_CopyKAT_call_HVG8000_r0_6.png
[PLOT] leiden_scvi_main
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_UMAP_leiden_scvi_main_HVG8000_r0_6.png
[PLOT] condition
[FIGURE] G:\Repository\glioma-c

In [11]:
# =========================
# BAR FIGURES
# =========================
plot_df = (
    obs[
        "final_compartment_major"
    ]
    .astype(str)
    .value_counts()
    .rename_axis(
        "final_compartment_major"
    )
    .reset_index(
        name="n_cells"
    )
)


plt.figure(
    figsize=(
        7,
        4,
    )
)

plt.bar(
    plot_df[
        "final_compartment_major"
    ],
    plot_df[
        "n_cells"
    ],
)

plt.xticks(
    rotation=45,
    ha="right",
)

plt.ylabel(
    "Number of cells"
)

plt.title(
    "Final compartment counts"
)

png, pdf = save_current_fig(
    "final_annotation_bar_final_compartment_counts_HVG8000_r0_6"
)

figure_manifest = pd.concat(
    [
        figure_manifest,
        pd.DataFrame(
            [
                {
                    "figure": "bar_final_compartment_counts",
                    "type": "bar",
                    "png": png,
                    "pdf": pdf,
                }
            ]
        ),
    ],
    ignore_index=True,
)


ct = pd.crosstab(
    obs[
        "final_compartment_major"
    ].astype(str),
    obs[
        COPYKAT_KEY
    ].astype(str),
)

prop = (
    ct.div(
        ct.sum(
            axis=1
        ),
        axis=0,
    )
    .fillna(
        0
    )
)


plt.figure(
    figsize=(
        7,
        4,
    )
)

bottom = np.zeros(
    prop.shape[
        0
    ]
)

x = np.arange(
    prop.shape[
        0
    ]
)


for column in prop.columns:
    plt.bar(
        x,
        prop[
            column
        ].values,
        bottom=bottom,
        label=column,
    )

    bottom += prop[
        column
    ].values


plt.xticks(
    x,
    prop.index,
    rotation=45,
    ha="right",
)

plt.ylabel(
    "Fraction"
)

plt.title(
    "CopyKAT composition by final compartment"
)

plt.legend(
    bbox_to_anchor=(
        1.02,
        1,
    ),
    loc="upper left",
)

png, pdf = save_current_fig(
    "final_annotation_bar_copykat_by_final_compartment_HVG8000_r0_6"
)

figure_manifest = pd.concat(
    [
        figure_manifest,
        pd.DataFrame(
            [
                {
                    "figure": "bar_copykat_by_final_compartment",
                    "type": "stacked_bar",
                    "png": png,
                    "pdf": pdf,
                }
            ]
        ),
    ],
    ignore_index=True,
)


write_tsv_replace(
    figure_manifest,
    FIGURE_MANIFEST_TSV,
)


if figure_manifest.shape != (
    9,
    5,
):
    raise RuntimeError(
        "Historical nine-figure manifest checkpoint failed: "
        f"{figure_manifest.shape} != (9, 5)"
    )


print(
    "PASS: historical bar-figure manifest reproduced."
)


[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_bar_final_compartment_counts_HVG8000_r0_6.png
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_bar_copykat_by_final_compartment_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidation_figure_manifest_HVG8000_r0_6.tsv shape=(9, 5)
PASS: historical bar-figure manifest reproduced.


In [12]:
# =========================
# MARKER DOTPLOT — MARKER-GENE SLICE ONLY
# =========================
print(
    "n marker genes present:",
    len(
        marker_genes_present
    ),
)


if marker_adata is not None:
    # Align final refined labels to the marker-only AnnData.
    if not np.array_equal(
        marker_adata.obs_names.astype(str),
        obs.index.astype(str),
    ):
        raise RuntimeError(
            "Marker-gene AnnData cell order does not match canonical obs order."
        )

    marker_adata.obs[
        "final_annotation_refined"
    ] = (
        obs[
            "final_annotation_refined"
        ]
        .copy()
    )

    if "counts" in marker_adata.layers:
        marker_adata.X = (
            marker_adata.layers[
                "counts"
            ]
            .copy()
        )

    elif "raw_counts" in marker_adata.layers:
        marker_adata.X = (
            marker_adata.layers[
                "raw_counts"
            ]
            .copy()
        )

    sc.pp.normalize_total(
        marker_adata,
        target_sum=1e4,
    )

    sc.pp.log1p(
        marker_adata
    )

    sc.pl.dotplot(
        marker_adata,
        var_names=marker_genes_present,
        groupby="final_annotation_refined",
        standard_scale="var",
        dendrogram=False,
        show=False,
    )

    png, pdf = save_current_fig(
        "final_annotation_marker_dotplot_by_refined_annotation_HVG8000_r0_6",
        dpi=180,
    )

    figure_manifest = pd.concat(
        [
            figure_manifest,
            pd.DataFrame(
                [
                    {
                        "figure": "marker_dotplot_by_refined_annotation",
                        "type": "dotplot",
                        "png": png,
                        "pdf": pdf,
                    }
                ]
            ),
        ],
        ignore_index=True,
    )

    write_tsv_replace(
        figure_manifest,
        FIGURE_MANIFEST_TSV,
    )

else:
    raise RuntimeError(
        "Historical marker dotplot requires the 77 present marker genes."
    )


if figure_manifest.shape != (
    10,
    5,
):
    raise RuntimeError(
        "Historical final figure-manifest checkpoint failed: "
        f"{figure_manifest.shape} != (10, 5)"
    )


print(
    "PASS: historical final ten-figure manifest reproduced exactly."
)


# Release marker-gene expression memory before writing the large H5AD.
del marker_adata
gc.collect()


n marker genes present: 77
[FIGURE] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_annotation_CNV_consensus_HVG8000_r0_6\final_annotation_marker_dotplot_by_refined_annotation_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidation_figure_manifest_HVG8000_r0_6.tsv shape=(10, 5)
PASS: historical final ten-figure manifest reproduced exactly.


486

In [13]:
# =========================
# BUILD / RECOVER FINAL H5AD — METADATA-ONLY VALIDATION
# =========================
final_annotation_consolidation = {
    "input_h5ad": str(
        INPUT_H5AD
    ),
    "cluster_key": CLUSTER_KEY,
    "copykat_key": COPYKAT_KEY,
    "previous_final_group_key": FINAL_GROUP_KEY,
    "new_columns": [
        "final_annotation_refined",
        "final_compartment_major",
        "final_compartment_simple",
        "final_analysis_use",
    ],
    "main_malignant_like_clusters": [
        "0",
        "4",
        "7",
        "11",
    ],
    "cycling_malignant_like_candidate_clusters": [
        "2"
    ],
    "reference_like_clusters": [
        "8",
        "14",
    ],
    "non_malignant_diploid_leaning_clusters": [
        "8",
        "13",
        "14",
        "15",
        "16",
    ],
    "review_clusters": [
        "1",
        "3",
        "5",
        "6",
        "9",
        "10",
        "12",
        "17",
    ],
}


output_obs = (
    obs
    .copy()
)


temp_h5ad = FINAL_H5AD.with_name(
    FINAL_H5AD.stem
    + ".building.h5ad"
)


generated_uns_updates = {
    "final_annotation_consolidation": (
        final_annotation_consolidation
    ),
}

generated_uns_updates.update(
    generated_color_uns
)


final_reused = False


if FINAL_H5AD.exists():
    print(
        "[RECOVERY CHECK] Existing final H5AD found:"
    )
    print(
        FINAL_H5AD
    )

    try:
        existing_final_audit = validate_h5ad_metadata_only(
            FINAL_H5AD,
            expected_shape=(
                351427,
                15176,
            ),
            expected_major_counts=EXPECTED_MAJOR_COUNTS,
            require_provenance=True,
        )

        print(
            "[RECOVERY PASS] Existing final H5AD is complete and canonical."
        )

        display(
            pd.DataFrame(
                [
                    existing_final_audit
                ]
            )
        )

        final_reused = True

    except Exception as error:
        print(
            "[RECOVERY FAIL] Existing final H5AD is incomplete or invalid:"
        )
        print(
            repr(
                error
            )
        )
        print(
            "[REBUILD] The invalid final H5AD will be replaced."
        )

        FINAL_H5AD.unlink()


if not final_reused:
    if temp_h5ad.exists():
        print(
            "[REMOVE STALE TEMP]",
            temp_h5ad,
        )

        temp_h5ad.unlink()


    source_size = (
        INPUT_H5AD
        .stat()
        .st_size
    )

    free_space = shutil.disk_usage(
        FINAL_H5AD.parent
    ).free


    print(
        "Source H5AD size GiB:",
        round(
            source_size
            / 1024**3,
            3,
        ),
    )

    print(
        "Free space GiB:",
        round(
            free_space
            / 1024**3,
            3,
        ),
    )


    if free_space < int(
        source_size
        * 1.05
    ):
        raise RuntimeError(
            "Insufficient free space for safe final-H5AD creation."
        )


    print(
        "[COPY CANONICAL NOTEBOOK-15 H5AD]"
    )

    copy_file_with_progress(
        INPUT_H5AD,
        temp_h5ad,
    )


    print(
        "[REPLACE /obs ONLY]"
    )

    replace_h5ad_obs_group(
        temp_h5ad,
        output_obs,
    )


    print(
        "[PATCH /uns]"
    )

    add_h5ad_uns_elements(
        temp_h5ad,
        generated_uns_updates,
    )


    print(
        "[METADATA-ONLY VALIDATION OF TEMP H5AD]"
    )

    temp_audit = validate_h5ad_metadata_only(
        temp_h5ad,
        expected_shape=(
            351427,
            15176,
        ),
        expected_major_counts=EXPECTED_MAJOR_COUNTS,
        require_provenance=True,
    )

    display(
        pd.DataFrame(
            [
                temp_audit
            ]
        )
    )


    os.replace(
        temp_h5ad,
        FINAL_H5AD,
    )

    print(
        "[SAVED]",
        FINAL_H5AD,
    )


# Final canonical validation never reads X or layers.
print(
    "[FINAL METADATA-ONLY VALIDATION]"
)

final_audit = validate_h5ad_metadata_only(
    FINAL_H5AD,
    expected_shape=(
        351427,
        15176,
    ),
    expected_major_counts=EXPECTED_MAJOR_COUNTS,
    require_provenance=True,
)

display(
    pd.DataFrame(
        [
            final_audit
        ]
    )
)


# ------------------------------------------------------------
# Compatibility alias
# ------------------------------------------------------------
# If the previous run already created the alias, validate and keep it.
# Otherwise create it as a hard link when possible.
alias_reused = False
alias_mode = None


if FINAL_ALIAS_H5AD.exists():
    print(
        "[RECOVERY CHECK] Existing alias found:"
    )
    print(
        FINAL_ALIAS_H5AD
    )

    try:
        alias_audit = validate_h5ad_metadata_only(
            FINAL_ALIAS_H5AD,
            expected_shape=(
                351427,
                15176,
            ),
            expected_major_counts=EXPECTED_MAJOR_COUNTS,
            require_provenance=True,
        )

        print(
            "[RECOVERY PASS] Existing alias is complete and canonical."
        )

        display(
            pd.DataFrame(
                [
                    alias_audit
                ]
            )
        )

        alias_reused = True

        try:
            alias_mode = (
                "existing_hard_link"
                if os.path.samefile(
                    FINAL_H5AD,
                    FINAL_ALIAS_H5AD,
                )
                else "existing_file"
            )

        except Exception:
            alias_mode = "existing_file"

    except Exception as error:
        print(
            "[RECOVERY FAIL] Existing alias is invalid:"
        )
        print(
            repr(
                error
            )
        )

        FINAL_ALIAS_H5AD.unlink()


if not alias_reused:
    alias_mode = create_alias(
        FINAL_H5AD,
        FINAL_ALIAS_H5AD,
    )

    alias_audit = validate_h5ad_metadata_only(
        FINAL_ALIAS_H5AD,
        expected_shape=(
            351427,
            15176,
        ),
        expected_major_counts=EXPECTED_MAJOR_COUNTS,
        require_provenance=True,
    )

    display(
        pd.DataFrame(
            [
                alias_audit
            ]
        )
    )


print(
    "Alias mode:",
    alias_mode,
)

print(
    "PASS: final H5AD and compatibility alias validated without reading X or layers."
)


gc.collect()


[RECOVERY CHECK] Existing final H5AD found:
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_CNV_consensus.h5ad
[RECOVERY PASS] Existing final H5AD is complete and canonical.


,path,shape,major_counts
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,"(351427, 15176)","{'review': 167178, 'malignant_like': 123250, '..."


[FINAL METADATA-ONLY VALIDATION]


,path,shape,major_counts
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,"(351427, 15176)","{'review': 167178, 'malignant_like': 123250, '..."


[RECOVERY CHECK] Existing alias found:
G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation.h5ad
[RECOVERY PASS] Existing alias is complete and canonical.


,path,shape,major_counts
0,G:\Repository\glioma-cnv-single-cell-atlas\dat...,"(351427, 15176)","{'review': 167178, 'malignant_like': 123250, '..."


Alias mode: existing_hard_link
PASS: final H5AD and compatibility alias validated without reading X or layers.


16527

In [14]:
# =========================
# OUTPUT MANIFEST + FINAL CHECKPOINTS
# =========================
manifest_items = [
    {
        "artifact": "readiness",
        "path": str(
            READINESS_TSV
        ),
        "exists": (
            READINESS_TSV.exists()
        ),
    },
    {
        "artifact": "cell_counts",
        "path": str(
            CELL_COUNTS_TSV
        ),
        "exists": (
            CELL_COUNTS_TSV.exists()
        ),
    },
    {
        "artifact": "cluster_summary",
        "path": str(
            CLUSTER_SUMMARY_TSV
        ),
        "exists": (
            CLUSTER_SUMMARY_TSV.exists()
        ),
    },
    {
        "artifact": "marker_presence",
        "path": str(
            MARKER_PRESENCE_TSV
        ),
        "exists": (
            MARKER_PRESENCE_TSV.exists()
        ),
    },
    {
        "artifact": "figure_manifest",
        "path": str(
            FIGURE_MANIFEST_TSV
        ),
        "exists": (
            FIGURE_MANIFEST_TSV.exists()
        ),
    },
    {
        "artifact": "final_h5ad",
        "path": str(
            FINAL_H5AD
        ),
        "exists": (
            FINAL_H5AD.exists()
        ),
    },
    {
        "artifact": "final_alias_h5ad",
        "path": str(
            FINAL_ALIAS_H5AD
        ),
        "exists": (
            FINAL_ALIAS_H5AD.exists()
        ),
    },
]


for key, path in composition_paths.items():
    manifest_items.append(
        {
            "artifact": key,
            "path": path,
            "exists": (
                Path(
                    path
                ).exists()
            ),
        }
    )


OUTPUT_MANIFEST_TSV = (
    METADATA_DIR
    / "final_annotation_consolidation_output_manifest_HVG8000_r0_6.tsv"
)

output_manifest = pd.DataFrame(
    manifest_items
)

write_tsv_replace(
    output_manifest,
    OUTPUT_MANIFEST_TSV,
)


if output_manifest.shape != (
    10,
    3,
):
    raise RuntimeError(
        "Historical final-annotation output-manifest checkpoint failed: "
        f"{output_manifest.shape} != (10, 3)"
    )


if not output_manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more expected Notebook-17 outputs are missing."
    )


display(
    output_manifest
)


print(
    "\nFINAL CHECKPOINTS"
)

print(
    "Canonical Notebook-15 input:",
    INPUT_H5AD,
)

print(
    "Source shape:",
    source_shape,
)

print(
    "Canonical CopyKAT counts:",
    EXPECTED_COPYKAT_COUNTS,
)

print(
    "Previous final malignancy-group counts:",
    EXPECTED_FINAL_GROUP_COUNTS,
)

print(
    "Final major-compartment counts:",
    EXPECTED_MAJOR_COUNTS,
)

print(
    "Refined annotation labels:",
    obs[
        "final_annotation_refined"
    ].nunique(),
)

print(
    "Readiness:",
    readiness.shape,
)

print(
    "Consolidated cell counts:",
    cell_counts_df.shape,
)

print(
    "Cluster summary:",
    cluster_summary.shape,
)

print(
    "Condition composition:",
    cond_counts.shape,
)

print(
    "Dataset composition:",
    dataset_counts.shape,
)

print(
    "Sample composition:",
    sample_counts.shape,
)

print(
    "Marker presence:",
    marker_presence.shape,
)

print(
    "Marker genes present:",
    int(
        marker_presence[
            "present"
        ].sum()
    ),
)

print(
    "Figure manifest:",
    figure_manifest.shape,
)

print(
    "Output manifest:",
    output_manifest.shape,
)

print(
    "Final H5AD:",
    FINAL_H5AD,
)

print(
    "Final alias H5AD:",
    FINAL_ALIAS_H5AD,
)

print(
    "Alias mode:",
    alias_mode,
)

print(
    "\n[DONE] Notebook 17 final annotation consolidation completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_annotation_consolidation_output_manifest_HVG8000_r0_6.tsv shape=(10, 3)


,artifact,path,exists
0,readiness,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
1,cell_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
2,cluster_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
3,marker_presence,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
4,figure_manifest,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
5,final_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
6,final_alias_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True
7,condition_compartment_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
8,dataset_compartment_counts,G:\Repository\glioma-cnv-single-cell-atlas\met...,True
9,sample_level_compartment_composition,G:\Repository\glioma-cnv-single-cell-atlas\met...,True



FINAL CHECKPOINTS
Canonical Notebook-15 input: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad
Source shape: (351427, 15176)
Canonical CopyKAT counts: {'aneuploid': 202895, 'diploid': 111493, 'not_defined': 37037, 'missing': 2}
Previous final malignancy-group counts: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Final major-compartment counts: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Refined annotation labels: 18
Readiness: (9, 2)
Consolidated cell counts: (57, 4)
Cluster summary: (18, 19)
Condition composition: (16, 5)
Dataset composition: (16, 5)
Sample composition: (319, 8)
Marker presence: (80, 3)
Marker genes present: 77
Figure manifest: (10, 5)
Output manifest: (10, 3)
Final H5AD: G:\Repository\glioma

## Expected historical checkpoints

A successful run should end with:

```text
Source shape: (351427, 15176)
Canonical CopyKAT counts:
    aneuploid 202895
    diploid 111493
    not_defined 37037
    missing 2

Previous final malignancy-group counts:
    review 167178
    malignant_like 123250
    cycling_malignant_like_candidate 38634
    non_malignant_or_diploid_leaning 22365

Final major-compartment counts: same four counts
Refined annotation labels: 18
Readiness: (9, 2)
Consolidated cell counts: (57, 4)
Cluster summary: (18, 19)
Condition composition: (16, 5)
Dataset composition: (16, 5)
Sample composition: (319, 8)
Marker presence: (80, 3)
Marker genes present: 77
Figure manifest: (10, 5)
Output manifest: (10, 3)
```

The two final H5AD paths are generated locally and should remain excluded from ordinary Git commits.